# 01. Data Exploration — ROLLINGMRP

This notebook loads the synthetic factory dataset and produces initial exploratory visualizations:
1. **Item Master Overview & Distribution** (Finished Goods, Subassemblies, Raw Materials)
2. **Bill of Materials (BOM) Tree Hierarchy & Commonality**
3. **Time-Phased MPS Demand Patterns** (Trend, Seasonality, Noise over 12 weeks)
4. **Starting Inventory & Asset Valuation**
5. **Work Center Capacity Profiles & Supplier Risk Distribution**

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
current_dir = Path.cwd().resolve()
repo_root = current_dir if (current_dir / "src").exists() else current_dir.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.integration.load import run_pipeline

# Set plot styles
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("Setup completed successfully. Project root:", repo_root)

## 1. Run Data Ingestion Pipeline

In [ ]:
data, quality_report = run_pipeline()
print("Data Ingestion Summary:")
for k, v in quality_report["summary"].items():
    print(f"  {k}: {v}")

items_df = data["items"]
bom_df = data["bom"]
inv_df = data["inventory_initialized"]
mps_df = data["mps"]
wc_df = data["work_centers"]
sup_df = data["suppliers"]

## 2. Item Master Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
type_counts = items_df["type"].value_counts()
bars = ax.bar(type_counts.index, type_counts.values, color=["#1f77b4", "#ff7f0e", "#2ca02c"])
ax.set_title("Distribution of Items by Classification", fontsize=14, fontweight="bold")
ax.set_ylabel("Number of Items")
ax.bar_label(bars, padding=3)
plt.tight_layout()
plt.show()

## 3. BOM Component Commonality Analysis
Show which components are shared across multiple parent items.

In [ ]:
component_usage = bom_df.groupby("component_item_id")["parent_item_id"].nunique().sort_values(ascending=False)
print("Top Shared Components (Multi-parent commonality):")
print(component_usage[component_usage > 1])

fig, ax = plt.subplots(figsize=(10, 4))
component_usage[component_usage > 1].plot(kind="bar", ax=ax, color="#9467bd")
ax.set_title("Component Commonality (Number of Parent Items Using Component)", fontsize=14, fontweight="bold")
ax.set_ylabel("Parent Count")
ax.set_xlabel("Component Item ID")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. 12-Week Time-Phased MPS Demand
Visualize demand for the 5 Finished Goods across the 12-week horizon.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for item_id, group in mps_df.groupby("item_id"):
    group_sorted = group.sort_values("period_number")
    ax.plot(group_sorted["period_number"], group_sorted["quantity"], marker="o", label=item_id, linewidth=2)

ax.set_title("12-Week MPS Demand Profiles by Finished Good", fontsize=14, fontweight="bold")
ax.set_xlabel("Planning Period (Weeks)")
ax.set_ylabel("Gross Demand (Units)")
ax.set_xticks(range(1, 13))
ax.legend(title="Finished Good", loc="upper left")
plt.tight_layout()
plt.show()

## 5. Inventory Status & Capital Valuation

In [ ]:
# Defensive check to ensure 'type' is present in inv_df
if "type" not in inv_df.columns:
    inv_df = inv_df.merge(items_df[["item_id", "type"]], on="item_id", how="left")

inv_summary = inv_df.groupby("type").agg({
    "quantity_on_hand": "sum",
    "net_available_start": "sum",
    "inventory_value": "sum"
})
print("Inventory Summary by Item Type:")
print(inv_summary.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
inv_summary["inventory_value"].plot(kind="bar", ax=ax, color="#2ca02c")
ax.set_title("Starting Inventory Valuation by Item Type ($)", fontsize=14, fontweight="bold")
ax.set_ylabel("Total Value ($)")
plt.tight_layout()
plt.show()

## 6. Work Center Capacities & Supplier Reliability

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Work Center Capacity
axes[0].bar(wc_df["work_center_id"], wc_df["capacity_hours_per_week"], color="#d62728")
axes[0].set_title("Weekly Work Center Available Capacity (Hours)", fontweight="bold")
axes[0].set_ylabel("Capacity Hours / Week")

# Supplier Reliability
axes[1].bar(sup_df["supplier_id"], sup_df["reliability_score"], color="#17becf")
axes[1].set_title("Supplier Reliability Ratings (Score: 0 - 1)", fontweight="bold")
axes[1].set_ylabel("Reliability Score")
axes[1].set_ylim(0.5, 1.05)

plt.tight_layout()
plt.show()
print("Data exploration complete. All data artifacts verified.")